# CERT 내부자 위협 데이터 분석

이 노트북은 CERT (Computer Emergency Response Team) 내부자 위협 데이터를 분석합니다.

## 목차
1. 데이터 로드 및 준비
2. 과거 위협 데이터 분석
3. 사용자 행동 분석
4. 시각화
5. 위협 탐지 모델

In [ ]:
# 필요한 라이브러리 임포트
import json
import csv
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (선택사항)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 임포트 완료")

## 1. 데이터 로드 및 준비

In [ ]:
# answers 폴더에서 정답 데이터 로드
with open('answers/threat_indicators.json', 'r', encoding='utf-8') as f:
    threat_data = json.load(f)
    
threat_indicators = threat_data['high_risk_indicators']
threat_scenarios = threat_data['threat_scenarios']
analysis_params = threat_data['analysis_parameters']

print("위협 지표 데이터 로드 완료")
print(f"\n분석 파라미터:")
print(json.dumps(analysis_params, indent=2, ensure_ascii=False))

In [ ]:
# 과거 위협 데이터 로드
known_threats_df = pd.read_csv('answers/known_threats.csv')
known_threats_df['detection_date'] = pd.to_datetime(known_threats_df['detection_date'])

print(f"과거 위협 데이터 로드 완료: {len(known_threats_df)} 건")
print("\n데이터 미리보기:")
known_threats_df.head()

## 2. 과거 위협 데이터 분석

In [ ]:
# 기본 통계
print("=" * 60)
print("과거 위협 데이터 통계")
print("=" * 60)
print(f"총 위협 건수: {len(known_threats_df)}")
print(f"\n위협 유형별 분포:")
print(known_threats_df['threat_type'].value_counts())
print(f"\n심각도별 분포:")
print(known_threats_df['severity'].value_counts())
print(f"\n해결 상태:")
print(known_threats_df['resolved'].value_counts())

In [ ]:
# 시각화 1: 위협 유형별 분포
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 위협 유형
threat_counts = known_threats_df['threat_type'].value_counts()
axes[0, 0].bar(range(len(threat_counts)), threat_counts.values)
axes[0, 0].set_xticks(range(len(threat_counts)))
axes[0, 0].set_xticklabels(threat_counts.index, rotation=45, ha='right')
axes[0, 0].set_title('Threat Type Distribution')
axes[0, 0].set_ylabel('Count')

# 심각도
severity_order = ['low', 'medium', 'high', 'critical']
severity_counts = known_threats_df['severity'].value_counts().reindex(severity_order, fill_value=0)
colors = ['green', 'yellow', 'orange', 'red']
axes[0, 1].bar(severity_order, severity_counts.values, color=colors)
axes[0, 1].set_title('Severity Distribution')
axes[0, 1].set_ylabel('Count')

# 월별 추이
known_threats_df['month'] = known_threats_df['detection_date'].dt.to_period('M')
monthly_counts = known_threats_df.groupby('month').size()
axes[1, 0].plot(range(len(monthly_counts)), monthly_counts.values, marker='o')
axes[1, 0].set_title('Monthly Threat Trend')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(True, alpha=0.3)

# 해결 상태
resolution_counts = known_threats_df['resolved'].value_counts()
axes[1, 1].pie(resolution_counts.values, labels=resolution_counts.index, autopct='%1.1f%%')
axes[1, 1].set_title('Resolution Status')

plt.tight_layout()
plt.savefig('cert_threat_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("시각화 저장 완료: cert_threat_analysis.png")

## 3. 사용자 행동 분석

In [ ]:
# 샘플 사용자 데이터 생성 (실제로는 데이터베이스나 로그에서 가져옴)
sample_users = pd.DataFrame([
    {
        'user_id': 'USR999',
        'daily_file_access': 1200,
        'unique_files': 600,
        'sensitive_files': 55,
        'external_emails': 120,
        'email_attachment_size_mb': 150,
        'off_hours_logins': 15,
        'failed_logins': 25,
        'data_transfer_gb': 60,
        'external_connections': 250
    },
    {
        'user_id': 'USR888',
        'daily_file_access': 500,
        'unique_files': 200,
        'sensitive_files': 10,
        'external_emails': 30,
        'email_attachment_size_mb': 20,
        'off_hours_logins': 2,
        'failed_logins': 3,
        'data_transfer_gb': 5,
        'external_connections': 50
    },
    {
        'user_id': 'USR777',
        'daily_file_access': 2000,
        'unique_files': 800,
        'sensitive_files': 100,
        'external_emails': 200,
        'email_attachment_size_mb': 300,
        'off_hours_logins': 30,
        'failed_logins': 50,
        'data_transfer_gb': 100,
        'external_connections': 400
    }
])

print("샘플 사용자 데이터:")
sample_users

In [ ]:
# 위험 점수 계산 함수
def calculate_risk_score(user_row):
    scores = {}
    
    # 파일 접근 점수
    file_score = 0
    if user_row['daily_file_access'] > threat_indicators['file_access']['threshold_daily_access']:
        file_score += 0.4
    if user_row['unique_files'] > threat_indicators['file_access']['threshold_unique_files']:
        file_score += 0.3
    if user_row['sensitive_files'] > threat_indicators['file_access']['threshold_sensitive_files']:
        file_score += 0.3
    scores['file_access'] = file_score
    
    # 이메일 점수
    email_score = 0
    if user_row['external_emails'] > threat_indicators['email_patterns']['external_recipients_threshold']:
        email_score += 0.5
    if user_row['email_attachment_size_mb'] > threat_indicators['email_patterns']['attachment_size_threshold_mb']:
        email_score += 0.5
    scores['email_activity'] = email_score
    
    # 로그인 점수
    login_score = 0
    if user_row['off_hours_logins'] > threat_indicators['login_patterns']['off_hours_login_threshold']:
        login_score += 0.5
    if user_row['failed_logins'] > threat_indicators['login_patterns']['failed_login_threshold']:
        login_score += 0.5
    scores['login_behavior'] = login_score
    
    # 네트워크 점수
    network_score = 0
    if user_row['data_transfer_gb'] > threat_indicators['network_activity']['data_transfer_threshold_gb']:
        network_score += 0.6
    if user_row['external_connections'] > threat_indicators['network_activity']['external_connection_threshold']:
        network_score += 0.4
    scores['network_activity'] = network_score
    
    # 가중 평균 계산
    weights = analysis_params['risk_score_weights']
    total_score = sum(scores[k] * weights[k] for k in scores.keys())
    
    return pd.Series({**scores, 'total_risk_score': total_score})

# 위험 점수 계산
risk_scores = sample_users.apply(calculate_risk_score, axis=1)
users_with_risk = pd.concat([sample_users, risk_scores], axis=1)

print("사용자별 위험 점수:")
users_with_risk[['user_id', 'file_access', 'email_activity', 'login_behavior', 'network_activity', 'total_risk_score']]

## 4. 위험 점수 시각화

In [ ]:
# 사용자별 위험 점수 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 전체 위험 점수 비교
axes[0].bar(users_with_risk['user_id'], users_with_risk['total_risk_score'])
axes[0].axhline(y=0.7, color='r', linestyle='--', label='Critical Threshold')
axes[0].axhline(y=0.5, color='orange', linestyle='--', label='High Threshold')
axes[0].axhline(y=0.3, color='yellow', linestyle='--', label='Medium Threshold')
axes[0].set_title('Total Risk Score by User')
axes[0].set_ylabel('Risk Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 카테고리별 위험 점수
categories = ['file_access', 'email_activity', 'login_behavior', 'network_activity']
x = np.arange(len(users_with_risk))
width = 0.2

for i, cat in enumerate(categories):
    axes[1].bar(x + i*width, users_with_risk[cat], width, label=cat)

axes[1].set_xlabel('User')
axes[1].set_ylabel('Risk Score')
axes[1].set_title('Risk Score by Category')
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(users_with_risk['user_id'])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('user_risk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("시각화 저장 완료: user_risk_analysis.png")

## 5. 위협 탐지 및 권장 조치

In [ ]:
# 위협 수준 분류 및 권장 조치
def classify_and_recommend(row):
    score = row['total_risk_score']
    
    if score >= 0.7:
        level = 'CRITICAL'
        action = 'Immediate investigation required'
    elif score >= 0.5:
        level = 'HIGH'
        action = 'Detailed review needed within 24 hours'
    elif score >= 0.3:
        level = 'MEDIUM'
        action = 'Monitor closely'
    elif score > 0:
        level = 'LOW'
        action = 'Regular monitoring'
    else:
        level = 'NORMAL'
        action = 'No action required'
    
    # 잠재적 위협 유형 식별
    threats = []
    if row['file_access'] > 0.5 and row['email_activity'] > 0.3:
        threats.append('Data Exfiltration')
    if row['login_behavior'] > 0.6:
        threats.append('Credential Theft')
    if row['file_access'] > 0.6:
        threats.append('Potential Sabotage')
    
    return pd.Series({
        'threat_level': level,
        'recommended_action': action,
        'potential_threats': ', '.join(threats) if threats else 'None'
    })

recommendations = users_with_risk.apply(classify_and_recommend, axis=1)
final_results = pd.concat([users_with_risk[['user_id', 'total_risk_score']], recommendations], axis=1)

print("="*80)
print("위협 분석 및 권장 조치")
print("="*80)
print(final_results.to_string())
print("="*80)

## 6. 결과 저장

In [ ]:
# 결과를 CSV 파일로 저장
output_file = 'cert_analysis_results.csv'
final_results.to_csv(output_file, index=False)
print(f"분석 결과가 {output_file}에 저장되었습니다.")

# 요약 통계
print("\n요약 통계:")
print(f"분석된 사용자 수: {len(final_results)}")
print(f"위협 수준별 분포:")
print(final_results['threat_level'].value_counts())